<a href="https://colab.research.google.com/github/abbanaish1-max/insurance-ruin-climate/blob/main/notebooks/01_oecd_Rt_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests

url = "https://sdmx.oecd.org/public/rest/dataflow/OECD.DAF.CM/DSD_INS@DF_BSI/1.0?references=all"

r = requests.get(url)
r.raise_for_status()

print(r.status_code)
print(r.text[:2000])

200
<?xml version="1.0" encoding="utf-8"?>
<!--NSI Web Service v8.19.8.0-->
<message:Structure xmlns:message="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/message" xmlns:structure="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/structure" xmlns:common="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/common">
  <message:Header>
    <message:ID>IDREF2258</message:ID>
    <message:Test>false</message:Test>
    <message:Prepared>2026-09-06T07:48:45.9829399+02:00</message:Prepared>
    <message:Sender id="Unknown" />
    <message:Receiver id="Unknown" />
  </message:Header>
  <message:Structures>
    <structure:Dataflows>
      <structure:Dataflow id="DSD_INS@DF_BSI" agencyID="OECD.DAF.CM" version="1.0" isFinal="true">
        <common:Annotations>
          <common:Annotation>
            <common:AnnotationType>NonProductionDataflow</common:AnnotationType>
            <common:AnnotationText xml:lang="en">true</common:AnnotationText>
          </common:Annotation>
          <common

In [ ]:
import requests
import xml.etree.ElementTree as ET
from collections import defaultdict

URL = (
    "https://sdmx.oecd.org/public/rest/dataflow/"
    "OECD.DAF.CM/DSD_INS@DF_BSI/1.0?references=all"
)

r = requests.get(URL, timeout=120)
r.raise_for_status()

print("HTTP status:", r.status_code)
print("Response size:", len(r.content), "bytes")

root = ET.fromstring(r.content)

# Remove XML namespaces for easier inspection
for elem in root.iter():
    elem.tag = elem.tag.split("}")[-1]
    for child in list(elem):
        child.tag = child.tag.split("}")[-1]

# ---------------------------------------------------------
# 1. List DataStructures
# ---------------------------------------------------------
print("\n=== DATA STRUCTURES ===")
for ds in root.iter("DataStructure"):
    print("ID:", ds.attrib.get("id"))
    print("Agency:", ds.attrib.get("agencyID"))
    print("Version:", ds.attrib.get("version"))

# ---------------------------------------------------------
# 2. List Codelists
# ---------------------------------------------------------
print("\n=== CODELISTS ===")

codelists = {}

for cl in root.iter("Codelist"):
    cl_id = cl.attrib.get("id")
    cl_name = cl.attrib.get("name")
    codelists[cl_id] = []

    print(f"\nCodelist: {cl_id}")
    print("Name:", cl_name)

    for code in cl.findall("Code"):
        code_id = code.attrib.get("id")
        name = ""

        for desc in code.iter("Name"):
            if desc.text:
                name = desc.text.strip()
                break

        codelists[cl_id].append((code_id, name))
        print(f"  {code_id:20s} -> {name}")

# ---------------------------------------------------------
# 3. Show dimensions and their codelist references
# ---------------------------------------------------------
print("\n=== DIMENSIONS ===")

for dimlist in root.iter("DimensionList"):

    for dim in list(dimlist):
        tag = dim.tag

        if tag not in ["Dimension", "TimeDimension"]:
            continue

        dim_id = dim.attrib.get("id")
        position = dim.attrib.get("position")

        print(f"\nDimension: {dim_id}")
        print("Position:", position)

        for enum in dim.iter("Enumeration"):
            ref = enum.find("Ref")
            if ref is not None:
                print(
                    "Codelist:",
                    ref.attrib.get("id"),
                    "Agency:",
                    ref.attrib.get("agencyID"),
                    "Version:",
                    ref.attrib.get("version")
                )

HTTP status: 200
Response size: 1521156 bytes

=== DATA STRUCTURES ===
ID: DSD_INS
Agency: OECD.DAF.CM
Version: 1.0

=== CODELISTS ===

Codelist: CL_AREA
Name: None
  AUS                  -> Australia
  AUT                  -> Austria
  BEL                  -> Belgium
  CAN                  -> Canada
  CHL                  -> Chile
  COL                  -> Colombia
  CRI                  -> Costa Rica
  CZE                  -> Czechia
  DNK                  -> Denmark
  EST                  -> Estonia
  FIN                  -> Finland
  FRA                  -> France
  DEU                  -> Germany
  GRC                  -> Greece
  HUN                  -> Hungary
  ISL                  -> Iceland
  IRL                  -> Ireland
  ISR                  -> Israel
  ITA                  -> Italy
  JPN                  -> Japan
  KOR                  -> Korea
  LVA                  -> Latvia
  LTU                  -> Lithuania
  LUX                  -> Luxembourg
  MEX                

In [ ]:
# ============================================================
# OECD INSURANCE CAPITAL / SURPLUS COVERAGE AUDIT
# Dataset: DSD_INS@DF_BSI
# Measure: EQU = Shareholders' equity
# Target period: 1970-2025
# ============================================================

import requests
import pandas as pd
from io import StringIO
from pathlib import Path

# ------------------------------------------------------------
# 1. OECD API configuration
# ------------------------------------------------------------

BASE = (
    "https://sdmx.oecd.org/public/rest/data/"
    "OECD.DAF.CM,DSD_INS@DF_BSI,1.0/"
)

# Dimension order from the metadata you retrieved:
#
# 1  REF_AREA
# 2  FREQ
# 3  MEASURE
# 4  UNIT_MEASURE
# 5  PREMIUMS
# 6  OWNERSHIP
# 7  INSURANCE_TYPE
# 8  INSURANCE_BUSINESS
# 9  INSURER_TYPE
# 10 CONTRACT_TYPE
# 11 EMPLOYER_TYPE
# 12 RISK_LOCATION
# 13 COUNTERPART_AREA
# 14 INSURANCE_CLASS
# 15 DESTINATION

# First four dimensions are specified.
# Remaining dimensions are wildcards so that we can inspect
# the complete availability of EQU before restricting further.

key_parts = (
    ["", "A", "EQU", "USD"]   # REF_AREA wildcard, Annual, Equity, USD
    + [""] * 11               # Dimensions 5-15 = wildcard
)

key = ".".join(key_parts)

url = BASE + key

params = {
    "startPeriod": "1970",
    "endPeriod": "2025",
    "dimensionAtObservation": "AllDimensions",
    "format": "csvfilewithlabels"
}

print("Query URL:")
print(url)
print()

# ------------------------------------------------------------
# 2. Download from official OECD API
# ------------------------------------------------------------

response = requests.get(url, params=params, timeout=180)

print("HTTP status:", response.status_code)
print("Downloaded bytes:", len(response.content))

response.raise_for_status()

# ------------------------------------------------------------
# 3. Read CSV
# ------------------------------------------------------------

df = pd.read_csv(StringIO(response.text))

print("\nRaw shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 4. Identify TIME_PERIOD column
# ------------------------------------------------------------

if "TIME_PERIOD" not in df.columns:
    raise ValueError(
        "TIME_PERIOD column not found. "
        "Inspect the returned columns above."
    )

df["TIME_PERIOD"] = pd.to_numeric(
    df["TIME_PERIOD"],
    errors="coerce"
)

df = df.dropna(subset=["TIME_PERIOD"]).copy()
df["TIME_PERIOD"] = df["TIME_PERIOD"].astype(int)

# ------------------------------------------------------------
# 5. Basic date coverage
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL EQUITY DATA COVERAGE")
print("=" * 70)

print("Earliest year returned :", df["TIME_PERIOD"].min())
print("Latest year returned   :", df["TIME_PERIOD"].max())
print("Number of observations :", len(df))

# ------------------------------------------------------------
# 6. Exact year availability
# ------------------------------------------------------------

target_years = set(range(1970, 2026))
available_years = set(df["TIME_PERIOD"].unique())

missing_years = sorted(target_years - available_years)

print("\nTarget period          : 1970-2025")
print("Expected annual years  : 56")
print("Available annual years :", len(target_years & available_years))

if missing_years:
    print("Missing years          :", missing_years)
else:
    print("Missing years          : NONE")

# ------------------------------------------------------------
# 7. Reference areas
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("REFERENCE AREAS WITH EQUITY DATA")
print("=" * 70)

if "REF_AREA" in df.columns:
    area_cols = ["REF_AREA"]

    if "Reference area" in df.columns:
        area_cols.append("Reference area")

    areas = (
        df[area_cols]
        .drop_duplicates()
        .sort_values("REF_AREA")
    )

    print(areas.to_string(index=False))
    print("\nNumber of reference areas:", areas["REF_AREA"].nunique())

# ------------------------------------------------------------
# 8. Relevant insurance dimensions
# ------------------------------------------------------------

for col in [
    "FREQ",
    "MEASURE",
    "UNIT_MEASURE",
    "INSURANCE_TYPE",
    "INSURANCE_BUSINESS",
    "INSURER_TYPE",
    "OWNERSHIP"
]:
    if col in df.columns:
        print("\n" + "=" * 70)
        print(f"{col}")
        print("=" * 70)
        print(df[col].drop_duplicates().tolist())

# ------------------------------------------------------------
# 9. Coverage by reference area
# ------------------------------------------------------------

if "REF_AREA" in df.columns:

    coverage = (
        df.groupby("REF_AREA")["TIME_PERIOD"]
        .agg(
            earliest="min",
            latest="max",
            n_years="nunique"
        )
        .reset_index()
    )

    coverage["continuous_1970_2025"] = (
        coverage["n_years"].eq(56)
        & coverage["earliest"].eq(1970)
        & coverage["latest"].eq(2025)
    )

    print("\n" + "=" * 70)
    print("REFERENCE-AREA COVERAGE")
    print("=" * 70)

    print(
        coverage
        .sort_values(
            ["continuous_1970_2025", "earliest"],
            ascending=[False, True]
        )
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 10. Check whether an actual WORLD aggregate exists
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WORLD AGGREGATE TEST")
print("=" * 70)

if "REF_AREA" in df.columns:

    world = df[df["REF_AREA"] == "W"]

    if world.empty:
        print("No REF_AREA = W observation returned.")
        print("Therefore, we must investigate aggregation across")
        print("reporting jurisdictions rather than assuming a world series.")
    else:
        print("WORLD observations found:", len(world))
        print(
            "World earliest:",
            world["TIME_PERIOD"].min()
        )
        print(
            "World latest:",
            world["TIME_PERIOD"].max()
        )
        print(
            "World years:",
            world["TIME_PERIOD"].nunique()
        )

# ------------------------------------------------------------
# 11. Save untouched API response
# ------------------------------------------------------------

out_dir = Path("oecd_rt_audit")
out_dir.mkdir(exist_ok=True)

raw_path = out_dir / "OECD_BSI_EQU_raw_1970_2025.csv"
df.to_csv(raw_path, index=False)

print("\n" + "=" * 70)
print("RAW DATA PRESERVATION")
print("=" * 70)
print("Saved:", raw_path)
print("IMPORTANT: This file is the unmodified OECD API extraction.")

# ------------------------------------------------------------
# 12. Final automated verdict
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("AUTOMATED PRELIMINARY VERDICT")
print("=" * 70)

if (
    df["TIME_PERIOD"].min() <= 1970
    and df["TIME_PERIOD"].max() >= 2025
    and not missing_years
):
    print("✅ EQU data exist across the full 1970-2025 requested period.")
    print("NEXT: identify the correct insurance/entity aggregation.")
else:
    print("⚠️ EQU does NOT have complete 1970-2025 coverage in this query.")
    print("NEXT: inspect the reference-area coverage and test the")
    print("OECD financial-balance-sheet dataset or another official source.")

Query URL:
https://sdmx.oecd.org/public/rest/data/OECD.DAF.CM,DSD_INS@DF_BSI,1.0/.A.EQU.USD...........

HTTP status: 200
Downloaded bytes: 2925255

Raw shape: (6600, 48)

Columns:
['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'REF_AREA', 'Reference area', 'FREQ', 'Frequency of observation', 'MEASURE', 'Measure', 'UNIT_MEASURE', 'Unit of measure', 'PREMIUMS', 'Premiums', 'OWNERSHIP', 'Ownership', 'INSURANCE_TYPE', 'Insurance type', 'INSURANCE_BUSINESS', 'Insurance business', 'INSURER_TYPE', 'Insurer type', 'CONTRACT_TYPE', 'Contract type', 'EMPLOYER_TYPE', 'Employer type', 'RISK_LOCATION', 'Risk location', 'COUNTERPART_AREA', 'Counterpart area', 'INSURANCE_CLASS', 'Insurance class', 'DESTINATION', 'Destination', 'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'OBS_STATUS', 'Observation status', 'CONF_STATUS', 'Confidentiality status', 'UNIT_MULT', 'Unit multiplier', 'CURRENCY', 'Currency', 'DECIMALS', 'Decimals']

OVERALL EQUITY DATA COVERAGE
Earliest year re

In [ ]:
# Clone the private GitHub repository
!git clone https://github.com/abbanaish1-max/insurance-ruin-climate.git

Cloning into 'insurance-ruin-climate'...
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
from google.colab import userdata
import subprocess

github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise RuntimeError(
        "GITHUB_TOKEN was not found in Colab Secrets."
    )

repo_url = "https://github.com/from google.colab import userdata
import subprocess

github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise RuntimeError(
        "GITHUB_TOKEN was not found in Colab Secrets."
    )

repo_url = "https://github.com/YOUR_GITHUB_USERNAME/insurance-ruin-climate.git"

auth_url = repo_url.replace(
    "https://github.com/",
    f"https://{github_token}@github.com/"
)

result = subprocess.run(
    ["git", "clone", auth_url],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Git clone failed.")

print("Repository cloned successfully.")/insurance-ruin-climate.git"

auth_url = repo_url.replace(
    "https://github.com/",
    f"https://{github_token}@github.com/"
)

result = subprocess.run(
    ["git", "clone", auth_url],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Git clone failed.")

print("Repository cloned successfully.")

SyntaxError: unterminated string literal (detected at line 11) (2876930462.py, line 11)

In [7]:
from google.colab import userdata
import subprocess

# Get GitHub token securely from Colab Secrets
github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise RuntimeError(
        "GITHUB_TOKEN was not found in Colab Secrets. "
        "Please create it first."
    )

# Your actual GitHub repository
repo_url = (
    "https://github.com/"
    "abbanaish1-max/"
    "insurance-ruin-climate.git"
)

# Create authenticated clone URL
auth_url = repo_url.replace(
    "https://github.com/",
    f"https://{github_token}@github.com/"
)

# Clone the private repository
result = subprocess.run(
    ["git", "clone", auth_url],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("GitHub repository cloning failed.")

print("✅ Repository cloned successfully.")

✅ Repository cloned successfully.


In [8]:
from pathlib import Path

repo = Path("/content/insurance-ruin-climate")

print("Repository exists:", repo.exists())
print("Repository path:", repo)

if repo.exists():
    print("\nTop-level contents:")
    for item in sorted(repo.iterdir()):
        print(" -", item.name)

Repository exists: True
Repository path: /content/insurance-ruin-climate

Top-level contents:
 - .git
 - .gitignore
 - README.md
 - notebooks


In [10]:
import sys

sys.path.insert(
    0,
    "/content/insurance-ruin-climate/src"
)

from archive_checkpoint import (
    create_checkpoint,
    sha256_file
)

print("✅ Research Archive Checkpoint module loaded.")

ModuleNotFoundError: No module named 'archive_checkpoint'

In [11]:
from pathlib import Path

repo = Path("/content/insurance-ruin-climate")

print("Repository exists:", repo.exists())

if not repo.exists():
    print("\n❌ Repository is not at /content/insurance-ruin-climate")
else:
    print("\n✅ Repository found")
    print("\nRepository contents:")
    for p in sorted(repo.rglob("*")):
        if p.is_file():
            print(p)

Repository exists: True

✅ Repository found

Repository contents:
/content/insurance-ruin-climate/.git/HEAD
/content/insurance-ruin-climate/.git/config
/content/insurance-ruin-climate/.git/description
/content/insurance-ruin-climate/.git/hooks/applypatch-msg.sample
/content/insurance-ruin-climate/.git/hooks/commit-msg.sample
/content/insurance-ruin-climate/.git/hooks/fsmonitor-watchman.sample
/content/insurance-ruin-climate/.git/hooks/post-update.sample
/content/insurance-ruin-climate/.git/hooks/pre-applypatch.sample
/content/insurance-ruin-climate/.git/hooks/pre-commit.sample
/content/insurance-ruin-climate/.git/hooks/pre-merge-commit.sample
/content/insurance-ruin-climate/.git/hooks/pre-push.sample
/content/insurance-ruin-climate/.git/hooks/pre-rebase.sample
/content/insurance-ruin-climate/.git/hooks/pre-receive.sample
/content/insurance-ruin-climate/.git/hooks/prepare-commit-msg.sample
/content/insurance-ruin-climate/.git/hooks/push-to-checkout.sample
/content/insurance-ruin-climate

In [12]:
import sys

src_path = "/content/insurance-ruin-climate/src"

if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Python path updated:")
print(src_path)

import archive_checkpoint

print("\n✅ archive_checkpoint imported successfully.")

Python path updated:
/content/insurance-ruin-climate/src


ModuleNotFoundError: No module named 'archive_checkpoint'

In [13]:
from pathlib import Path

repo = Path("/content/insurance-ruin-climate")
src = repo / "src"

print("Repository exists:", repo.exists())
print("src exists:", src.exists())

print("\nFiles currently inside src:")
if src.exists():
    for item in sorted(src.iterdir()):
        print(" -", item.name)

print("\nLooking specifically for archive_checkpoint.py:")
matches = list(repo.rglob("archive_checkpoint.py"))

if matches:
    for m in matches:
        print("FOUND:", m)
else:
    print("❌ archive_checkpoint.py NOT FOUND")

Repository exists: True
src exists: False

Files currently inside src:

Looking specifically for archive_checkpoint.py:
FOUND: /content/insurance-ruin-climate/notebooks/src/archive_checkpoint.py


In [14]:
from pathlib import Path
import shutil

repo = Path("/content/insurance-ruin-climate")

if repo.exists():
    shutil.rmtree(repo)

print("Old repository clone removed.")

Old repository clone removed.


In [15]:
from google.colab import userdata
import subprocess

github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise RuntimeError(
        "GITHUB_TOKEN was not found in Colab Secrets."
    )

repo_url = (
    "https://github.com/"
    "abbanaish1-max/"
    "insurance-ruin-climate.git"
)

auth_url = repo_url.replace(
    "https://github.com/",
    f"https://{github_token}@github.com/"
)

result = subprocess.run(
    ["git", "clone", auth_url],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Repository clone failed.")

print("✅ Corrected repository cloned.")

✅ Corrected repository cloned.


In [16]:
from pathlib import Path

repo = Path("/content/insurance-ruin-climate")

print("Repository exists:", repo.exists())
print("src exists:", (repo / "src").exists())

print("\nTop-level directories:")
for item in sorted(repo.iterdir()):
    if item.is_dir() and item.name != ".git":
        print(" -", item.name)

print("\nFiles in src:")
src = repo / "src"

if src.exists():
    for item in sorted(src.iterdir()):
        print(" -", item.name)

print("\nChecking for misplaced module:")
wrong = repo / "notebooks" / "src" / "archive_checkpoint.py"

print(
    "Wrong location exists:",
    wrong.exists()
)

print(
    "Correct location exists:",
    (src / "archive_checkpoint.py").exists()
)

Repository exists: True
src exists: False

Top-level directories:
 - notebooks

Files in src:

Checking for misplaced module:
Wrong location exists: True
Correct location exists: False


In [17]:
from pathlib import Path
import shutil

repo = Path("/content/insurance-ruin-climate")

if repo.exists():
    shutil.rmtree(repo)

print("✅ Old repository clone removed.")

✅ Old repository clone removed.


In [18]:
from google.colab import userdata
import subprocess

github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise RuntimeError(
        "GITHUB_TOKEN was not found in Colab Secrets."
    )

repo_url = (
    "https://github.com/"
    "abbanaish1-max/"
    "insurance-ruin-climate.git"
)

auth_url = repo_url.replace(
    "https://github.com/",
    f"https://{github_token}@github.com/"
)

result = subprocess.run(
    ["git", "clone", auth_url],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Repository clone failed.")

print("✅ Corrected repository cloned successfully.")

✅ Corrected repository cloned successfully.


In [19]:
from pathlib import Path

repo = Path("/content/insurance-ruin-climate")
src = repo / "src"

print("Repository exists:", repo.exists())
print("src exists:", src.exists())

print("\nTop-level folders:")
for item in sorted(repo.iterdir()):
    if item.is_dir() and item.name != ".git":
        print(" -", item.name)

print("\nFiles in src:")
if src.exists():
    for item in sorted(src.iterdir()):
        print(" -", item.name)

wrong = repo / "notebooks" / "src" / "archive_checkpoint.py"
correct = src / "archive_checkpoint.py"

print("\nWrong location exists:", wrong.exists())
print("Correct location exists:", correct.exists())

Repository exists: True
src exists: False

Top-level folders:
 - notebooks

Files in src:

Wrong location exists: True
Correct location exists: False


In [1]:
from pathlib import Path
import shutil

repo = Path("/content/insurance-ruin-climate")

# Remove an older clone if one exists
if repo.exists():
    shutil.rmtree(repo)

print("Old repository copy removed.")

Old repository copy removed.


In [2]:
!git clone https://github.com/abbanaish1-max/insurance-ruin-climate.git

Cloning into 'insurance-ruin-climate'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 29 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 11.01 KiB | 3.67 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [3]:
from pathlib import Path

repo = Path("/content/insurance-ruin-climate")

print("Repository exists:", repo.exists())

print("\nTop-level contents:")
for item in sorted(repo.iterdir()):
    print(" -", item.name)

Repository exists: True

Top-level contents:
 - .git
 - .gitignore
 - README.md
 - metadata
 - notebooks
 - src


In [4]:
checkpoint_module = (
    repo / "src" / "archive_checkpoint.py"
)

print(
    "Checkpoint module exists:",
    checkpoint_module.exists()
)

print(
    "Path:",
    checkpoint_module
)

Checkpoint module exists: True
Path: /content/insurance-ruin-climate/src/archive_checkpoint.py


In [5]:
import sys

src_path = str(repo / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

from archive_checkpoint import (
    create_checkpoint,
    sha256_file
)

print("✅ Research Archive Checkpoint module loaded successfully.")

✅ Research Archive Checkpoint module loaded successfully.


In [6]:
from pathlib import Path

test_file = Path("/content/archive_test.txt")

test_file.write_text(
    """Research Archive Checkpoint Test

Project:
Climate-Driven Insurance Ruin Risk

Purpose:
Testing the archival infrastructure.

Status:
Infrastructure test only.
""",
    encoding="utf-8"
)

print("Test file created:", test_file)

Test file created: /content/archive_test.txt


In [7]:
checkpoint = create_checkpoint(
    stage="archive_infrastructure_test",
    files=[
        test_file
    ],
    metadata={
        "project": "Climate-Driven Insurance Ruin Risk",
        "stage": "Infrastructure test",
        "purpose": "Verify checkpoint generation and file integrity",
        "source": "Internal test file",
        "dataset_status": "Not research data",
        "redistribution_status": "Not applicable",
    },
    archive_root="/content/research_archive",
)

print("✅ Checkpoint created:")
print(checkpoint)

✅ Checkpoint created:
/content/research_archive/20260907T064002Z_archive_infrastructure_test


In [8]:
from pathlib import Path

checkpoint = Path(checkpoint)

print("Checkpoint contents:")

for item in sorted(checkpoint.rglob("*")):
    print(" -", item.relative_to(checkpoint))

Checkpoint contents:
 - files
 - files/archive_test.txt
 - manifest.json
 - metadata.json


In [9]:
import json

manifest_path = checkpoint / "manifest.json"

manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)

print(json.dumps(
    manifest,
    indent=2,
    ensure_ascii=False
))

{
  "checkpoint_created_utc": "20260907T064002Z",
  "stage": "archive_infrastructure_test",
  "metadata": {
    "project": "Climate-Driven Insurance Ruin Risk",
    "stage": "Infrastructure test",
    "purpose": "Verify checkpoint generation and file integrity",
    "source": "Internal test file",
    "dataset_status": "Not research data",
    "redistribution_status": "Not applicable"
  },
  "files": [
    {
      "filename": "archive_test.txt",
      "size_bytes": 160,
      "sha256": "0810ab06f3a2c250ba4e82c5b55657ea9485c64bc1c4f6b104c89850dbf69376"
    }
  ]
}


In [10]:
import shutil

archived_file = (
    checkpoint
    / "files"
    / "archive_test.txt"
)

recovered_file = Path(
    "/content/archive_test_recovered.txt"
)

shutil.copy2(
    archived_file,
    recovered_file
)

original_hash = sha256_file(
    test_file
)

recovered_hash = sha256_file(
    recovered_file
)

print("Original SHA-256 :", original_hash)
print("Recovered SHA-256:", recovered_hash)
print(
    "MATCH:",
    original_hash == recovered_hash
)

Original SHA-256 : 0810ab06f3a2c250ba4e82c5b55657ea9485c64bc1c4f6b104c89850dbf69376
Recovered SHA-256: 0810ab06f3a2c250ba4e82c5b55657ea9485c64bc1c4f6b104c89850dbf69376
MATCH: True
